In [ ]:
# 04 – Pathway & Gene Set Enrichment Analysis

#Input:
#- Symbol-level expression matrix (`expr_symbol`)
#- Sample metadata (`metadata`)
#- Differential expression table (`de_late_vs_early_sym`), late (≥168h) vs early (0h)

#Goal:
#- Extract up/down gene lists
#- Run enrichment (GO, KEGG, Hallmark, etc.)
#- Make nice summary plots (top pathways, immune signatures)

In [10]:
import pandas as pd

expr_symbol = pd.read_parquet("data/processed/expr_symbol.parquet")
metadata    = pd.read_csv("data/processed/metadata_aligned.csv", index_col=0)
de_late_vs_early_sym = pd.read_csv("results/de_late_vs_early_sym.csv", index_col=0)

expr_symbol.shape, metadata.shape, de_late_vs_early_sym.shape


ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - Missing optional dependency 'pyarrow'. pyarrow is required for parquet support. Use pip or conda to install pyarrow.
 - Missing optional dependency 'fastparquet'. fastparquet is required for parquet support. Use pip or conda to install fastparquet.

In [4]:
# 1) Load (or reuse) your raw matrix with ENSEMBL IDs as index
expr_with_ids = expr_matrix.copy()   # or whatever variable holds the raw counts

# 2) Strip version numbers from ENSEMBL IDs
expr_with_ids["ensembl_clean"] = expr_with_ids.index.to_series().str.split(".").str[0]

# 3) Map ENSEMBL → gene symbol
expr_with_ids["gene_symbol"] = expr_with_ids["ensembl_clean"].map(ensg_to_symbol)

# 4) Drop unmapped genes
expr_mapped = expr_with_ids.dropna(subset=["gene_symbol"])

# 5) Collapse duplicate symbols (multiple ENSEMBL IDs → same symbol)
expr_symbol = (
    expr_mapped
    .drop(columns=["ensembl_clean"])  # optional housekeeping
    .groupby("gene_symbol")
    .mean()
)

expr_symbol.head()
expr_df = expr_matrix  # alias for convenience


NameError: name 'expr_matrix' is not defined

In [5]:
ens_clean = expr_matrix.index.to_series().str.split(".").str[0]
symbols   = ens_clean.map(ensg_to_symbol)

expr_symbol = (
    expr_matrix
    .assign(gene_symbol=symbols)
    .dropna(subset=["gene_symbol"])
    .groupby("gene_symbol")
    .mean()
)

expr_symbol.head()


NameError: name 'expr_matrix' is not defined

In [6]:
expr_df = expr_symbol

early_mask = metadata["hours"] == 0
late_mask  = metadata["hours"] >= 168

de_late_vs_early_sym = differential_expression(
    expr_df, metadata,
    mask_group1=early_mask,
    mask_group2=late_mask,
    group1_name="early_0h",
    group2_name="late_168hplus"
)

de_late_vs_early_sym.head()


NameError: name 'expr_symbol' is not defined

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

# You can use gseapy for enrichment
try:
    import gseapy as gp
except ImportError:
    !pip install gseapy
    import gseapy as gp

pd.set_option("display.max_rows", 20)
pd.set_option("display.max_columns", 20)


In [ ]:
# Adjust paths to however you organized the repo
expr_symbol = pd.read_parquet("data/processed/expr_symbol.parquet")
metadata    = pd.read_csv("data/processed/metadata_aligned.csv", index_col=0)
de_late_vs_early_sym = pd.read_csv("results/de_late_vs_early_sym.csv", index_col=0)

expr_symbol.shape, metadata.shape, de_late_vs_early_sym.shape
